In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# 1. Data preparation and split (70% train, 30% test)
def prepare_data():
    # Load raw data
    data = pd.read_excel('dataCARS.xlsx')

    # Separate features and target
    X = data.iloc[:, :-1]  # all columns except the last are features
    y = data.iloc[:, -1]   # the last column is the chemical value (target)

    # Split into training and test sets (70% train, 30% test)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42
    )

    # Save to Excel files
    train_data = pd.concat([X_train, y_train], axis=1)
    test_data = pd.concat([X_test, y_test], axis=1)

    train_data.to_excel('train-1.xlsx', index=False, sheet_name='Sheet1')
    test_data.to_excel('test-1.xlsx', index=False, sheet_name='Sheet1')

    print(f"Training set size: {X_train.shape}")
    print(f"Test set size: {X_test.shape}")

    return X_train, X_test, y_train, y_test, X, y

# 2. LSSVM model construction and training
def train_lssvm_model(X_train, X_test, y_train, y_test):
    # Feature standardization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Create an LSSVM-like model (SVR with RBF kernel)
    lssvm_model = SVR(
        kernel='rbf',
        C=1.0,
        gamma='scale',
        epsilon=0.1
    )

    # Train the model
    lssvm_model.fit(X_train_scaled, y_train)

    # Save the model and the scaler
    joblib.dump(lssvm_model, 'lssvm_model.pkl')
    joblib.dump(scaler, 'scaler.pkl')

    # Evaluate the model
    y_train_pred = lssvm_model.predict(X_train_scaled)
    y_test_pred = lssvm_model.predict(X_test_scaled)

    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)

    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)

    # Save results to file only (no printing)
    results = {
        'train_rmse': train_rmse,
        'train_r2': train_r2,
        'test_rmse': test_rmse,
        'test_r2': test_r2
    }

    with open('lssvm_results.json', 'w') as f:
        import json
        json.dump(results, f, indent=2)

    return lssvm_model, scaler, X_train_scaled, X_test_scaled

# 3. Generate SHAP-compatible analysis script
def generate_shap_compatible_code():
    shap_code = """
# SHAP analysis script - compatible with the LSSVM model

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
import shap

# Load the LSSVM model and the scaler
lssvm_model = joblib.load('lssvm_model.pkl')
scaler = joblib.load('scaler.pkl')

# Load training and test sets
data_train = pd.read_excel('train-1.xlsx', sheet_name='Sheet1', header=0)
data_test = pd.read_excel('test-1.xlsx', sheet_name='Sheet1', header=0)

X_train = data_train.iloc[:, :-1].values
y_train = data_train.iloc[:, -1].values

X_test = data_test.iloc[:, :-1].values
y_test = data_test.iloc[:, -1].values

# Standardize the features
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model predictions
y_pred_train = lssvm_model.predict(X_train_scaled)
y_pred_test = lssvm_model.predict(X_test_scaled)

# Visualize training and test results
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(y_train, label='True')
plt.plot(y_pred_train, label='Predicted')
plt.title('Training Set')
plt.xlabel('Sample')
plt.ylabel('Value')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(y_test, label='True')
plt.plot(y_pred_test, label='Predicted')
plt.title('Test Set')
plt.xlabel('Sample')
plt.ylabel('Value')
plt.legend()

plt.tight_layout()
plt.show()

# Compute performance metrics
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_r2 = r2_score(y_train, y_pred_train)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
test_r2 = r2_score(y_test, y_pred_test)

print('Training Set - RMSE:', train_rmse, 'R2:', train_r2)
print('Test Set - RMSE:', test_rmse, 'R2:', test_r2)

# SHAP analysis
shap.initjs()
explainer = shap.KernelExplainer(lssvm_model.predict, X_train_scaled[:100])

sample_indices = [0, 10, 20]
for idx in sample_indices:
    shap_values = explainer.shap_values(X_test_scaled[idx:idx+1])

    shap.force_plot(
        explainer.expected_value,
        shap_values[0],
        X_test_scaled[idx],
        feature_names=data_train.columns[:-1].tolist(),
        matplotlib=True
    )
    plt.title(f'SHAP Force Plot - Sample {idx}')
    plt.show()

    shap.summary_plot(
        shap_values,
        X_test_scaled[idx:idx+1],
        feature_names=data_train.columns[:-1].tolist()
    )
    plt.show()

shap_values_all = explainer.shap_values(X_test_scaled[:50])

shap.summary_plot(
    shap_values_all,
    X_test_scaled[:50],
    feature_names=data_train.columns[:-1].tolist(),
    plot_type="bar"
)
plt.show()

shap.summary_plot(
    shap_values_all,
    X_test_scaled[:50],
    feature_names=data_train.columns[:-1].tolist()
)
plt.show()
"""

    with open('shap_analysis_lssvm.py', 'w', encoding='utf-8') as f:
        f.write(shap_code)

    print("SHAP analysis script saved as 'shap_analysis_lssvm.py'")

# 4. Main execution function
def main():
    print("Preparing data...")
    X_train, X_test, y_train, y_test, X_full, y_full = prepare_data()

    print("\nTraining LSSVM model...")
    model, scaler, X_train_scaled, X_test_scaled = train_lssvm_model(
        X_train, X_test, y_train, y_test
    )

    print("\nGenerating SHAP-compatible code...")
    generate_shap_compatible_code()

    print("\nDone! Run 'shap_analysis_lssvm.py' to start the SHAP analysis.")

if __name__ == "__main__":
    main()